In [1]:
import platform
import matplotlib.pyplot as plt

# OS에 따른 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')  # 윈도우: 맑은 고딕
elif platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')    # 맥: 애플 고딕
else:
    plt.rc('font', family='NanumBarunGothic') # 리눅스

# 마이너스 기호 깨짐 방지
plt.rc('axes', unicode_minus=False)

In [2]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

load_dotenv()

API_KEY = os.getenv("nvidiaapi_key")
MODEL = "openai/gpt-oss-20b"

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY)


@tool
def send_email(to: str, content: str) -> str:
    """이메일을 전송합니다."""

    print("받는 사람 : ", to)
    print("내용 : ", content)

    return f"{to}에게 이메일을 성공적으로 보냈습니다."


checkpointer = InMemorySaver()

hitl = HumanInTheLoopMiddleware(interrupt_on={"send_email": True})

agent = create_agent(
    model=llm, tools=[send_email], middleware=[hitl], checkpointer=checkpointer
)

config = {"configurable": {"thread_id": "email_test_001"}}

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "김철수에게 '회의가 오후 3시에 있습니다.' 라고 이메일 보내줘.",
            }
        ]
    },
    config=config,
)

interrupts = result.get("__interrupt__")

print("interrupts")

decision = input("\n이메일을 보내겠습니까? (y = 승인 / n = 거절) : ")

if decision.lower() == "y":
    resume_command = Command(resume={"decisions": [{"type": "approve"}]})
else:
    resume_command = Command(
        resume={
            "decisions": [
                {"type": "reject", "message": "사용자가 이메일 발송을 거절했습니다."}
            ]
        }
    )

result = agent.invoke(resume_command, config=config)


for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)

interrupts
HumanMessage
김철수에게 '회의가 오후 3시에 있습니다.' 라고 이메일 보내줘.
AIMessage
죄송하지만 이메일을 보낼 수 있는 정확한 주소를 알려주실 수 있나요? 현재 **김철수** 님의 이메일 주소를 확인할 수 없어서 전달이 어렵습니다. 주소를 알려주시면 바로 전송해드리겠습니다.


In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

load_dotenv(override=True)

# 1. 모델 준비
model = ChatOpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("nvidiaapi_key"),
    model="openai/gpt-oss-20b",
)

# 2. 이메일을 가려주는 미들웨어를 하나만 붙인 에이전트
agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="mask", apply_to_input=True),
    ],
)

# 3. 실행
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "제 이메일은 test@example.com 입니다."}
    ]
})

print(result["messages"][0].content)   # 가려진 입력 확인
print(result["messages"][-1].content)  # 에이전트 응답 확인

제 이메일은 test@****.com 입니다.
네, 이메일을 확인했습니다. 혹시 다른 도움이 필요하신가요?


In [4]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = [
    Document(
        page_content="""
        환불규정
        상품 구매 후 7일 이내에는 환불을 요청할 수 있습니다.
                단, 상품을 사용하거나 훼손한 경우 환불이 제한될 수 있습니다.
        
                배송 규정
        
                상품은 결제 완료 후 영업일 기준 2~3일 이내 배송됩니다.
                """
    )
]

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100,
)

chunks = splitter.split_documents(documents)

for i, chunk in enumerate(chunks):
    print(f"{i} : {chunk.page_content}")

c:\sk-encoa\llm_workspace\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


0 : 환불규정
        상품 구매 후 7일 이내에는 환불을 요청할 수 있습니다.
                단, 상품을 사용하거나 훼손한 경우 환불이 제한될 수 있습니다.

                배송 규정

                상품은 결제 완료 후 영업일 기준 2~3일 이내 배송됩니다.


In [5]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = [
    Document(
        page_content="""
        환불 규정

        상품 구매 후 7일 이내에는 환불을 요청할 수 있습니다.
        단, 상품을 사용하거나 훼손한 경우 환불이 제한될 수 있습니다.

        배송 규정

        상품은 결제 완료 후 영업일 기준 2~3일 이내 배송됩니다.
        """
    )
]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

chunks = splitter.split_documents(documents)

for i, chunk in enumerate(chunks):
    print(f"{i} : {chunk.page_content}")

0 : 환불 규정

        상품 구매 후 7일 이내에는 환불을 요청할 수 있습니다.
        단, 상품을 사용하거나 훼손한 경우 환불이 제한될 수 있습니다.

        배송 규정

        상품은 결제 완료 후 영업일 기준 2~3일 이내 배송됩니다.


In [6]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

load_dotenv()

PDF_PATH = "./data/금융투자협회_투자길라잡이_2018.pdf"

API_KEY = os.getenv("nvidiaapi_key")
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"

embedding = NVIDIAEmbeddings(api_key=API_KEY, model=EMBEDDING_MODEL)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

print("documents : ", len(documents))

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

print("chunk : ", len(chunks))

vectorstore = FAISS.from_documents(documents=chunks, embedding=embedding)

print("FIASS 생성완료")

query = "펀드가 무엇인가요?"

result = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(result):
    print("결과 : ", i)
    print("페이지 : ", doc.metadata.get("page"))
    print()
    print(doc.page_content)
    print()

C:\Users\playdata2\AppData\Local\Temp\ipykernel_12316\3061236395.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\sk-encoa\llm_workspace\.venv\Lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


documents :  191
chunk :  332
FIASS 생성완료
결과 :  0
페이지 :  31

것보다는 장기적인 안목을 가지고 결정하는 것이 바람직합니다. 판매회사에서 보내주는 각종 보고서의 정보를 
참고하여 내 펀드의 현재 상태를 점검하고 금융시장의 현황을 수시로 관찰하면서 향후 경제 전망 정보 등을 
참고하여 적절한 환매시점을 결정합니다. 
•펀드 환매절차 : 환매를 신청하고자 할 경우 환매시 적용되는 기준가격, 환매대금지급일, 환매수수료(환매제한
기간중에는 투자자가 환매수수료를 부담)에 관한 정보를 미리 확인하여야 합니다. 이 정보들은 투자설명서를 
통해 확인 가능합니다. 투자설명서는 펀드 가입시 제공되며, 금융투자협회의 인터넷 전자공시서비스 페이지
(dis.kofia.or.kr)에서도 확인할 수 있습니다. 환매는 직접 지점방문, 전화 또는 인터넷 사이트를 통해 신청할 수 
있으며, 환매대금을 지정한 계좌로 입금 또는 창구에서 현금으로 직접 지급받을 수 있습니다.
Q.
A.
펀드도 예금자보호대상이 되나요?
펀드는 예금이 아니라 실적배당상품이므로 펀드에 가입된 투자자들의 자금은 예금자보호대상이 아닙니다. 
그러나 펀드의 경우 투자자들의 자금으로 취득한 펀드재산은 자산운용회사의 고유재산과 분리되어 
신탁업자가 별도로 관리하기 때문에 자산운용회사가 파산하더라도 펀드내의 집합투자재산은 안전하다고 
할 수 있습니다.

결과 :  1
페이지 :  28

ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
   분쟁조정사례·판례집28
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
▒ 펀드 가입전 정보수집
가. 펀드 상품의 특성 이해 
•원금손실 발생 가능 : 펀드는 원금이 보장되지 않는 실적배당형 상품입니다. 운용결과에 따라 수익이 제한없이
늘어날 수 있지만, 원금손실이 발생할 수도 있다는 점에 유의하여야 합니다.
•펀드상품의 장점 : 일정 수준까지 원금 및 이자를 보장해주는 은행권의 예·적금에 비해 안정성 측면에서는
불리 하지만, 운용성과에 따라 높은 수익을 기

RecursivCharacterTextSplitter 및 Vextor DB로 LLM까지 연결

In [7]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

API_KEY = os.getenv("nvidiaapi_key")
MODEL = "openai/gpt-oss-20b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"

PDF_PATH = "./data/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatNVIDIA(api_key=API_KEY, model=MODEL)

embeddings = NVIDIAEmbeddings(api_key=API_KEY, model=EMBEDDING_MODEL)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)

retriever = vectorstore.as_retriever(search_kwargs={"k":3})

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

            반드시 아래 제공된 문서 내용만 근거로 답변하세요.

            문서에 답이 없다면
            "제공된 문서에서 답을 찾을 수 없습니다."
            라고 답변하세요.

            [문서 내용]
            {context}
            """,
        ),
        (
            "user",
            "[질문]{question}",
        ),
    ]
)

parser = StrOutputParser()

# question = "펀드란 무엇인가요?"
question = "대한민국의 수도는 어디인가요?"

retriever_docs = retriever.invoke(question)

context = "\n\n".join([doc.page_content for doc in retriever_docs])

chain = prompt | llm | parser

result = chain.invoke({"context": context, "question": question})

print("result:", result)


c:\sk-encoa\llm_workspace\.venv\Lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


result: 제공된 문서에서 답을 찾을 수 없습니다.


Vector DB에서 similarity_search_with_score를 통한 유사도 점수 확인

In [8]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

API_KEY = os.getenv("nvidiaapi_key")
MODEL = "openai/gpt-oss-20b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"
PDF_PATH = "./data/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatNVIDIA(model = MODEL, api_key = API_KEY)

embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, api_key=API_KEY)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)

questions = [
    "펀드란 무엇인가요?",
    "대한민국의 수도는 어디인가요?",
]

for question in questions:
    result = vectorstore.similarity_search_with_score(question, k=3)

    print("질문 : ", question)

    for i, (doc, score) in enumerate(result):
        print("검색결과")
        print("distance : ", score)
        print("page : ", doc.metadata.get("page"))
        print("내용 : ", doc.page_content[:500])

c:\sk-encoa\llm_workspace\.venv\Lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


질문 :  펀드란 무엇인가요?
검색결과
distance :  1.2219471
page :  31
내용 :  것보다는 장기적인 안목을 가지고 결정하는 것이 바람직합니다. 판매회사에서 보내주는 각종 보고서의 정보를 
참고하여 내 펀드의 현재 상태를 점검하고 금융시장의 현황을 수시로 관찰하면서 향후 경제 전망 정보 등을 
참고하여 적절한 환매시점을 결정합니다. 
•펀드 환매절차 : 환매를 신청하고자 할 경우 환매시 적용되는 기준가격, 환매대금지급일, 환매수수료(환매제한
기간중에는 투자자가 환매수수료를 부담)에 관한 정보를 미리 확인하여야 합니다. 이 정보들은 투자설명서를 
통해 확인 가능합니다. 투자설명서는 펀드 가입시 제공되며, 금융투자협회의 인터넷 전자공시서비스 페이지
(dis.kofia.or.kr)에서도 확인할 수 있습니다. 환매는 직접 지점방문, 전화 또는 인터넷 사이트를 통해 신청할 수 
있으며, 환매대금을 지정한 계좌로 입금 또는 창구에서 현금으로 직접 지급받을 수 있습니다.
Q.
A.
펀드도 예금자보호대상이 되나요?
펀드는 예금이 아니라 실적배당상품이므로 펀드에 가입된 투자자들의 
검색결과
distance :  1.2478452
page :  28
내용 :  ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
   분쟁조정사례·판례집28
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
▒ 펀드 가입전 정보수집
가. 펀드 상품의 특성 이해 
•원금손실 발생 가능 : 펀드는 원금이 보장되지 않는 실적배당형 상품입니다. 운용결과에 따라 수익이 제한없이
늘어날 수 있지만, 원금손실이 발생할 수도 있다는 점에 유의하여야 합니다.
•펀드상품의 장점 : 일정 수준까지 원금 및 이자를 보장해주는 은행권의 예·적금에 비해 안정성 측면에서는
불리 하지만, 운용성과에 따라 높은 수익을 기대할 수 있다는 장점이 있습니다.
나. 본인의 투자성향, 목표 등의 확인 
•나의 투자성향과 목표는? : 펀드에 가입할 때에는 높은 수익만을 기대하고 무작정 가입하는 것보다는 본인의
투자성향, 투자